In [ ]:
!pip install -q colpali-engine==0.3.10 byaldi pdf2image faiss-cpu Pillow tqdm
!pip install -q torchvision --upgrade
!apt-get install -q poppler-utils

print('Done installing — RESTARTING RUNTIME NOW...')
import os
os.kill(os.getpid(), 9)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 766.6/766.6 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.1/150.1 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.7/188.7 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import os
from google.colab import files

os.makedirs('data/pdfs', exist_ok=True)
os.makedirs('data/index/thumbnails', exist_ok=True)

print('Select your PDF files to upload (hold Ctrl to select multiple):')
uploaded = files.upload()

for filename, content in uploaded.items():
    path = f'data/pdfs/{filename}'
    with open(path, 'wb') as f:
        f.write(content)
    size_mb = len(content) / 1e6
    print(f'  Saved: {path} ({size_mb:.1f} MB)')

print(f'\nTotal PDFs uploaded: {len(uploaded)}')

Select your PDF files to upload (hold Ctrl to select multiple):


Saving attention_paper.pdf to attention_paper.pdf
Saving colpali_paper.pdf to colpali_paper.pdf
Saving esa_space_environment.pdf to esa_space_environment.pdf
Saving gpt4_paper.pdf to gpt4_paper.pdf
Saving mistral_paper.pdf to mistral_paper.pdf
  Saved: data/pdfs/attention_paper.pdf (2.2 MB)
  Saved: data/pdfs/colpali_paper.pdf (8.3 MB)
  Saved: data/pdfs/esa_space_environment.pdf (7.8 MB)
  Saved: data/pdfs/gpt4_paper.pdf (5.2 MB)
  Saved: data/pdfs/mistral_paper.pdf (27.5 MB)

Total PDFs uploaded: 5


In [ ]:
import json
from dataclasses import dataclass, asdict
from pathlib import Path
from pdf2image import convert_from_path
from PIL import Image
from tqdm import tqdm

@dataclass
class PageRecord:
    doc_id: int
    pdf_name: str
    page_number: int
    total_pages: int
    image_path: str

    @property
    def citation(self):
        return f"{self.pdf_name}, p. {self.page_number}"

pages = []
images = []
doc_id = 0
MAX_PAGES_PER_PDF = 20  # limit to save time

pdf_dir = Path('data/pdfs')
thumb_dir = Path('data/index/thumbnails')
thumb_dir.mkdir(parents=True, exist_ok=True)

for pdf_path in sorted(pdf_dir.glob('*.pdf')):
    print(f'Processing: {pdf_path.name}')
    try:
        pdf_images = convert_from_path(str(pdf_path), dpi=120, fmt='RGB')
        pdf_images = pdf_images[:MAX_PAGES_PER_PDF]
        print(f'  {len(pdf_images)} pages')

        for page_idx, img in enumerate(pdf_images):
            page_number = page_idx + 1
            thumb_name = f"{pdf_path.stem}_p{page_number:04d}.jpg"
            thumb_path = thumb_dir / thumb_name
            thumb = img.copy()
            thumb.thumbnail((800, 1200), Image.LANCZOS)
            thumb.save(str(thumb_path), 'JPEG', quality=85)

            pages.append(PageRecord(
                doc_id=doc_id,
                pdf_name=pdf_path.stem,
                page_number=page_number,
                total_pages=len(pdf_images),
                image_path=str(thumb_path),
            ))
            images.append(img)
            doc_id += 1

    except Exception as e:
        print(f'  FAILED: {e}')

# Save metadata
with open('data/index/metadata.json', 'w') as f:
    json.dump([asdict(p) for p in pages], f, indent=2)

print(f'\nTotal pages: {len(pages)}')

Processing: attention_paper.pdf
  15 pages
Processing: colpali_paper.pdf
  20 pages
Processing: esa_space_environment.pdf
  20 pages
Processing: gpt4_paper.pdf
  20 pages
Processing: mistral_paper.pdf
  20 pages

Total pages: 95


In [ ]:
import gc
import pickle
import numpy as np
import torch
import faiss
from tqdm import tqdm
from colpali_engine.models import ColPali, ColPaliProcessor

# Check GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Load ColPali
print('\nLoading ColPali model...')
model = ColPali.from_pretrained(
    'vidore/colpali-v1.3',
    torch_dtype=torch.float16,
    device_map=device,
).eval()
processor = ColPaliProcessor.from_pretrained('vidore/colpali-v1.3')
print('Model loaded ✓')

Device: cuda
GPU: Tesla T4
VRAM: 15.6 GB

Loading ColPali model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


adapter_config.json:   0%|          | 0.00/751 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

preprocessor_config.json:   0%|          | 0.00/423 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/733 [00:00<?, ?B/s]

Model loaded ✓


In [ ]:
# Encode all pages
BATCH_SIZE = 4 if device == 'cuda' else 1
all_embeddings = []

print(f'Encoding {len(images)} pages (batch_size={BATCH_SIZE})...')

with torch.no_grad():
    for i in tqdm(range(0, len(images), BATCH_SIZE)):
        batch = images[i:i+BATCH_SIZE]
        inputs = processor.process_images(batch)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        embeddings = model(**inputs)

        for emb in embeddings:
            arr = emb.cpu().float().numpy()
            norms = np.linalg.norm(arr, axis=1, keepdims=True) + 1e-8
            all_embeddings.append(arr / norms)

        del inputs, embeddings
        gc.collect()
        if device == 'cuda':
            torch.cuda.empty_cache()

print(f'Encoded {len(all_embeddings)} pages ✓')

Encoding 95 pages (batch_size=4)...


100%|██████████| 24/24 [00:52<00:00,  2.17s/it]

Encoded 95 pages ✓


In [ ]:
# Build FAISS index
print('Building FAISS index...')

page_map = []
patch_counts = []

for page_id, patch_embs in enumerate(all_embeddings):
    patch_counts.append(len(patch_embs))
    page_map.extend([page_id] * len(patch_embs))

matrix = np.vstack(all_embeddings).astype(np.float32)
dim = matrix.shape[1]
print(f'Matrix: {matrix.shape}')

index = faiss.IndexFlatIP(dim)
index.add(matrix)
print(f'FAISS index: {index.ntotal} vectors ✓')

# Save index
faiss.write_index(index, 'data/index/colpali.faiss')

meta = {
    'page_map': page_map,
    'patch_counts': patch_counts,
    'model_name': 'vidore/colpali-v1.3',
    'embedding_dim': dim,
}
with open('data/index/index_meta.pkl', 'wb') as f:
    pickle.dump(meta, f)

print('Index saved ✓')

Building FAISS index...
Matrix: (97945, 128)
FAISS index: 97945 vectors ✓
Index saved ✓


In [ ]:
import shutil
from google.colab import files

# Zip the entire index folder
print('Zipping index folder...')
shutil.make_archive('colpali_index', 'zip', 'data/index')

size_mb = os.path.getsize('colpali_index.zip') / 1e6
print(f'colpali_index.zip: {size_mb:.1f} MB')

print('Downloading...')
files.download('colpali_index.zip')
print('Done! Extract the zip and place contents into your local data/index/ folder')

Zipping index folder...
colpali_index.zip: 61.3 MB
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Extract the zip and place contents into your local data/index/ folder
